In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType,
    TimestampType, DateType, BooleanType
)

In [0]:
VELOCITY_THRESHOLD  = 10
LARGE_TXN_THRESHOLD = 5000
NIGHT_OWL_THRESHOLD = 3
NIGHT_HOURS         = (0, 5)
CROSS_BORDER_PCT    = 0.50

In [0]:
SILVER_SOURCE = "banking_demo.silver.transactions_cleansed"
silver_txns = spark.read.table(SILVER_SOURCE)

SILVER_SOURCE = "banking_demo.silver.fx_rates_silver"
silver_fx = spark.read.table(SILVER_SOURCE)

In [0]:
silver_txns.groupBy("event_date") \
    .agg(
        F.count("*").alias("total_txn_count"),
        F.coalesce(F.round(F.sum("gbp_amount"), 2), F.lit(0.0)).alias("total_gbp_volume"),
        F.countDistinct("customer_id").alias("distinct_customers"),
        F.countDistinct("account_id").alias("distinct_accounts"),
        F.sum(F.when(F.col("is_high_risk_merchant") == True, 1).otherwise(0)).alias("high_risk_txn_count"),
        F.coalesce(F.round(F.sum(F.when(F.col("is_high_risk_merchant") == True, F.col("gbp_amount"))), 2), F.lit(0.0)).alias("high_risk_gbp_volume"),
        F.sum(F.when(F.col("is_cross_border") == True, 1).otherwise(0)).alias("cross_border_count"),
        F.coalesce(F.round(F.sum(F.when(F.col("is_cross_border") == True, F.col("gbp_amount"))), 2), F.lit(0.0)).alias("cross_border_gbp_volume"),
        F.sum(F.when(F.col("counterparty_sanctioned") == True, 1).otherwise(0)).alias("sanctioned_count"),
        F.sum(F.when(F.col("txn_hour").between(NIGHT_HOURS[0], NIGHT_HOURS[1]), 1).otherwise(0)).alias("late_night_count"),
        F.sum(F.when(F.col("txn_type") == "REFUND", 1).otherwise(0)).alias("refund_count"),
        F.coalesce(F.round(F.avg("gbp_amount"), 2), F.lit(0.0)).alias("avg_txn_gbp_value"),
        F.coalesce(F.round(F.max("gbp_amount"), 2), F.lit(0.0)).alias("max_single_txn_gbp"),
    ) \
    .withColumn("refund_rate_pct", F.round(F.col("refund_count") / F.col("total_txn_count") * 100, 4)) \
    .withColumn("cross_border_pct", F.col("cross_border_count") / F.col("total_txn_count")) \
    .withColumn("risk_score",
        F.when(F.col("sanctioned_count") > 0, F.lit(100)).otherwise(F.lit(0)) +
        F.when(F.col("high_risk_txn_count") > F.col("total_txn_count") * 0.10, F.lit(30)).otherwise(F.lit(0)) +
        F.when(F.col("cross_border_pct") > 0.30, F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("late_night_count") > F.col("total_txn_count") * 0.05, F.lit(15)).otherwise(F.lit(0)) +
        F.when(F.col("refund_rate_pct") > 5.0, F.lit(10)).otherwise(F.lit(0)) +
        F.when(F.col("max_single_txn_gbp") > 10000, F.lit(10)).otherwise(F.lit(0))
    ) \
    .withColumn("daily_risk_band",
        F.when(F.col("risk_score") >= 100, F.lit("CRITICAL"))
        .when(F.col("risk_score") >= 50, F.lit("HIGH"))
        .when(F.col("risk_score") >= 20, F.lit("MEDIUM"))
        .otherwise(F.lit("LOW"))
    ) \
    .drop("cross_border_pct", "risk_score") \
    .withColumn("gold_processed_ts", F.current_timestamp()) \
    .withColumn("pipeline_version", F.lit("1.0.0")).display()


In [0]:
summary   = spark.read.table("daily_risk_summary")
anomalies = spark.read.table("daily_anomaly_flags")
fx_exp    = spark.read.table("daily_fx_exposure")

In [0]:
counts = (
    summary.agg(
        F.count("*").alias("summary_days"),
        F.coalesce(
            F.round(F.sum("total_gbp_volume"), 2),
            F.lit(0.0)).alias("total_pipeline_gbp_volume"),
        F.sum(F.when(
            F.col("daily_risk_band").isin("HIGH","CRITICAL"), 1)
            .otherwise(0)).alias("high_risk_days"),
        F.sum("sanctioned_count")
            .alias("total_sanctioned_hits"),
    )
    .crossJoin(anomalies.agg(
        F.count("*").alias("anomaly_rows"),
        F.sum(F.when(F.col("any_flag") == True, 1)
                .otherwise(0)).alias("flagged_customers"),
        F.sum(F.when(F.col("sanctioned_flag") == True, 1)
                .otherwise(0)).alias("sanctioned_customers"),
    ))
    .crossJoin(fx_exp.agg(
        F.count("*").alias("fx_exposure_rows"),
        F.sum(F.when(F.col("rate_risk_flag") == True, 1)
                .otherwise(0)).alias("fx_risk_flag_count"),
    ))
    .withColumn("audit_status",
        F.when(
            (F.col("summary_days") == 0) |
            (F.col("total_sanctioned_hits") > 0) |
            (F.col("sanctioned_customers") > 0),
            F.lit("FAIL")
        ).otherwise(F.lit("PASS")))
)

In [0]:
counts.display()

In [0]:




row = counts.collect()[0]

risk_band_dist = (
    summary.groupBy("daily_risk_band")
            .agg(F.count("*").alias("count"))
            .orderBy("daily_risk_band")
            .toPandas()
            .set_index("daily_risk_band")["count"]
            .to_dict()
)

In [0]:


    counts = (
        aml.agg(
            F.count("*").alias("aml_rows"),
            F.sum(F.when(
                F.col("regulatory_status") == "REPORT", 1)
                .otherwise(0)).alias("sar_count"),
            F.sum(F.when(
                F.col("regulatory_status") == "REVIEW", 1)
                .otherwise(0)).alias("review_count"),
            F.sum(F.when(
                F.col("sanctions_hit") == True, 1)
                .otherwise(0)).alias("sanctions_count"),
        )
        .crossJoin(basel.agg(
            F.count("*").alias("basel_rows"),
            F.sum(F.when(
                F.col("capital_adequacy_band") == "BREACH", 1)
                .otherwise(0)).alias("breach_count"),
            F.sum(F.when(
                F.col("capital_adequacy_band") == "WATCH", 1)
                .otherwise(0)).alias("watch_count"),
        ))
        .withColumn("audit_status",
            F.when(
                (F.col("aml_rows") == 0) |
                (F.col("basel_rows") == 0) |
                (F.col("sanctions_count") > 0) |
                (F.col("breach_count") > 0),
                F.lit("FAIL")
            ).otherwise(F.lit("PASS")))
    )

    row = counts.collect()[0]

    latest_log = (
        log.orderBy(F.desc("run_ts"))
           .limit(1)
           .select("run_id", "report_hash", "status")
           .collect()[0]
    )